In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
# import constants for the days of the week
from matplotlib.dates import MO, TU, WE, TH, FR, SA, SU
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec

## Importing Streamflow Data

In [ ]:
rme_discharge = pd.read_excel('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Streamflow/reymnte2025_Discharge.xls')
rme_discharge

In [ ]:
def fix_discharge_date(series):
    if series['time'] == '2400':
        series['time'] = '0000'
        series['Date'] = series['Date'] + pd.Timedelta('1d')
    return series

rme_discharge['time'] = rme_discharge['time'].astype(str).str.zfill(4)
rme_discharge = rme_discharge.apply(fix_discharge_date, axis=1)
rme_discharge['Datetime'] = pd.to_datetime(rme_discharge['Date'].astype(str) + ' ' + rme_discharge['time'], errors='coerce')
rme_discharge.set_index('Datetime', inplace=True)
rme_discharge.sort_index(inplace=True)
rme_discharge = rme_discharge.resample('15min').ffill()
rme_discharge


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))

rme_discharge.loc['10/1/2024':'11/30/2025'].plot(y='qcfs', logy=True, title = 'RME Discharge 1/23/25 - 05/12/2025', ax=ax)
#ax.invert_yaxis()

# Importing SCAN Data

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True)

# Importing Grab Sample Data

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/'
rme_results = pd.read_csv(result_dir+'rme_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
rme_results = rme_results.drop(['2/15/25 3:30:00', '2-22-25 11:00']) # drop outliers
# Filter out VOL flagged results
rme_results_novol = rme_results[~rme_results['Nitrate QA'].str.contains('VOL')]
rme_results_vol = rme_results[rme_results['Nitrate QA'].str.contains('VOL')]

rme_results

In [ ]:
rme_results_novol['2025/07/01':]

## Converting Sample Data to Timeseries

In [ ]:
rme_results_novol.index = rme_results_novol.index.round('15min')
rme_results_15 = rme_results_novol[['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].reset_index().groupby('Sample Datetime').mean().resample('15min').interpolate('time')

In [ ]:
rme_results_15.plot(y=['Nitrate mean', 'Phosphate mean', 'Ammonium mean'])

# Merge Scan, Grab Sample and Discharge Data

In [ ]:
rme_export = pd.merge_asof(rme_discharge, rme_cleaned, left_index=True, right_index=True, direction='nearest')
rme_export = pd.merge_asof(rme_export, rme_results_15, left_index=True, right_index=True, direction='nearest')
#rme_export = rme_export.loc['1/23/2025':]


# Calculate Export

In [ ]:
def calculate_export(df, discharge_col, area, conc_cols):
    for col in conc_cols:
        df[col + ' Export'] = 25485.12/1e6/area*df[discharge_col]*df[col]
        df[col + ' Cumulative Export'] = df[col + ' Export'].cumsum()
    return df



In [ ]:
rme_export = calculate_export(rme_export, 'qcfs', 36, ['second_derivative_no3_mgl_bias_correct', 'Nitrate mean', 'Ammonium mean'])
rme_export['TIN Cumulative Export'] = rme_export['second_derivative_no3_mgl_bias_correct Cumulative Export'] + rme_export['Ammonium mean Cumulative Export']

In [ ]:
rme_export.plot(y=['second_derivative_no3_mgl_bias_correct Cumulative Export', 'Nitrate mean Cumulative Export', 'Ammonium mean Cumulative Export', 'TIN Cumulative Export'], figsize=(11, 6))

# Export

In [ ]:
rme_export['10/1/2024':].to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_export.csv')
rme_discharge['10/1/24':].to_excel('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rme_discharge.xlsx')
